In [1]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [4]:
class TrackedObject:
  def __init__(self, **kwargs):
    # history сам добавляем через super().__setattr__, чтобы не ловить в логе
    super().__setattr__('history', [])
    for key, value in kwargs.items():
      setattr(self, key, value)

  def __setattr__(self, name, value):
    old_has = name in self.__dict__
    old_len = len(self.__dict__) if hasattr(self, '__dict__') else 0
    # выполняем операцию
    super().__setattr__(name, value)
    
    new_len = len(self.__dict__)
    self.history.append({
        'operation': 'set',
        'name': name,
        'was_existing': old_has,
        'dict_len_before': old_len,
        'dict_len_after': new_len
    })

  def __delattr__(self, name):
    old_len = len(self.__dict__) if hasattr(self, '__dict__') else 0
    
    # выполняем операцию
    super().__delattr__(name)
    
    new_len = len(self.__dict__)
    self.history.append({
        'operation': 'set',
        'name': name,
        'dict_len_before': old_len,
        'dict_len_after': new_len
    })

obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x
obj.y = 4

print(obj.__dict__)
for event in obj.history:
  print(event)

{'history': [{'operation': 'set', 'name': 'x', 'was_existing': False, 'dict_len_before': 1, 'dict_len_after': 2}, {'operation': 'set', 'name': 'y', 'was_existing': False, 'dict_len_before': 2, 'dict_len_after': 3}, {'operation': 'set', 'name': 'z', 'was_existing': False, 'dict_len_before': 3, 'dict_len_after': 4}, {'operation': 'set', 'name': '__str__', 'was_existing': False, 'dict_len_before': 4, 'dict_len_after': 5}, {'operation': 'set', 'name': 'x', 'dict_len_before': 5, 'dict_len_after': 4}, {'operation': 'set', 'name': 'y', 'was_existing': True, 'dict_len_before': 4, 'dict_len_after': 4}], 'y': 4, 'z': 3, '__str__': <function <lambda> at 0x7a5b6c3a4400>}
{'operation': 'set', 'name': 'x', 'was_existing': False, 'dict_len_before': 1, 'dict_len_after': 2}
{'operation': 'set', 'name': 'y', 'was_existing': False, 'dict_len_before': 2, 'dict_len_after': 3}
{'operation': 'set', 'name': 'z', 'was_existing': False, 'dict_len_before': 3, 'dict_len_after': 4}
{'operation': 'set', 'name': '__

## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [ ]:
def merge(seqs):
    result = []
    seqs = [s for s in seqs if s]
    while seqs:
        for seq in seqs:
            head = seq[0]
            if all(head not in s[1:] for s in seqs):
                result.append(head)
                seqs = [s[1:] if s and s[0] == head else s for s in seqs]
                seqs = [s for s in seqs if s]
                break
        else:
            raise ValueError("Inconsistent hierarchy")
    return result

def c3_linearize(cls):
    if not cls.__bases__:
        return [cls]
    return [cls] + merge([c3_linearize(b) for b in cls.__bases__] + [list(cls.__bases__)])

class A: 
    def action(self): return "A"

class B(A): 
    def action(self): return "B"

class C(A): 
    def action(self): return "C"  # Conflict: B and C both have action()

class D(B, C): 
    def action(self): return "D"  # Resolves the conflict

class E(D): 
    pass

class F(E): 
    pass

# Проверка
for cls in [A, B, C, D, E, F]:
    my = [c.__name__ for c in c3_linearize(cls)]
    real = [c.__name__ for c in cls.__mro__]
    print(f"{cls.__name__}: {my}== {real} {my == real}")


# Formula: L(C) = C + merge(L(P1), L(P2), ..., [P1, P2, ...])
# And rules:
# Local precedence: Parents appear in the order they are listed in class definition
# Monotonicity: Once X comes before Y, this order never reverses in subclasses
# No duplicates: Each class appears exactly once
# Ensure that conflicts are minimal and order is defined



A: ['A', 'object']== ['A', 'object'] True
B: ['B', 'A', 'object']== ['B', 'A', 'object'] True
C: ['C', 'A', 'object']== ['C', 'A', 'object'] True
D: ['D', 'B', 'C', 'A', 'object']== ['D', 'B', 'C', 'A', 'object'] True
E: ['E', 'D', 'B', 'C', 'A', 'object']== ['E', 'D', 'B', 'C', 'A', 'object'] True
F: ['F', 'E', 'D', 'B', 'C', 'A', 'object']== ['F', 'E', 'D', 'B', 'C', 'A', 'object'] True


## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [13]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "base_secret"  # name mangling: _SecureBase__secret_value
        self._semi_private = "base_semi"
        self.public = "base_public"

class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()
        self.__secret_value = "child_secret"  # name mangling: _SecureChild__secret_value
        self._semi_private = "child_semi"

    def dump_dict(self):
        return self.__dict__


base = SecureBase()
child = SecureChild()

print("dir()")
print("Base dir:", [d for d in dir(base) if 'secret' in d or 'semi' in d or 'public' in d])
print("Child dir:", [d for d in dir(child) if 'secret' in d or 'semi' in d or 'public' in d])

print("\n__dict__ ")
print(f"Base __dict__: {base.__dict__}")
print(f"Child __dict__: {child.__dict__}")
print(f"Child dump_dict(): {child.dump_dict()}")

print("\n Real storage names ")
print(f"Base: {[k for k in base.__dict__.keys()]}")
print(f"Child: {[k for k in child.__dict__.keys()]}")

print("\n Access mangled from outside ")
print(f"Base secret via mangled: {base._SecureBase__secret_value}")
print(f"Child secret via mangled: {child._SecureChild__secret_value}")
print(f"Child base secret via mangled: {child._SecureBase__secret_value}")

dir()
Base dir: ['_SecureBase__secret_value', '_semi_private', 'public']
Child dir: ['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private', 'public']

__dict__ 
Base __dict__: {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'base_semi', 'public': 'base_public'}
Child __dict__: {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'child_semi', 'public': 'base_public', '_SecureChild__secret_value': 'child_secret'}
Child dump_dict(): {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'child_semi', 'public': 'base_public', '_SecureChild__secret_value': 'child_secret'}

 Real storage names 
Base: ['_SecureBase__secret_value', '_semi_private', 'public']
Child: ['_SecureBase__secret_value', '_semi_private', 'public', '_SecureChild__secret_value']

 Access mangled from outside 
Base secret via mangled: base_secret
Child secret via mangled: child_secret
Child base secret via mangled: base_secret


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [14]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ("x", "y")
    
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")
    
    def __init__(self, x):
        self.x = x

def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d

n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

for obj in [n, w, ww]:
    print(describe_instance(obj))

# Попытка динамического добавления атрибутов
try:
    n.z = 100
    print(f"NoSlots: добавлен z = {n.z}")
except AttributeError as e:
    print(f"NoSlots: {e}")

try:
    w.z = 100
    print(f"WithSlots: добавлен z = {w.z}")
except AttributeError as e:
    print(f"WithSlots: {e}")

try:
    ww.z = 100
    print(f"WithSlotsWeak: добавлен z = {ww.z}")
except AttributeError as e:
    print(f"WithSlotsWeak: {e}")

# weakref
try:
    weak_n = weakref.ref(n)
    print(f"NoSlots: weakref создан, объект жив: {weak_n() is not None}")
except TypeError as e:
    print(f"NoSlots: {e}")

try:
    weak_w = weakref.ref(w)
    print(f"WithSlots: weakref создан, объект жив: {weak_w() is not None}")
except TypeError as e:
    print(f"WithSlots: {e}")

try:
    weak_ww = weakref.ref(ww)
    print(f"WithSlotsWeak: weakref создан, объект жив: {weak_ww() is not None}")
except TypeError as e:
    print(f"WithSlotsWeak: {e}")

{'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y']}
{'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
{'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}
NoSlots: добавлен z = 100
WithSlots: 'WithSlots' object has no attribute 'z'
WithSlotsWeak: 'WithSlotsWeak' object has no attribute 'z'
NoSlots: weakref создан, объект жив: True
WithSlots: cannot create weak reference to 'WithSlots' object
WithSlotsWeak: weakref создан, объект жив: True


## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [15]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        self.parent = None
        if parent:
            if weak_parent:
                self.parent = weakref.ref(parent)  # слабая ссылка
            else:
                self.parent = parent  # сильная ссылка

    def get_parent(self):
        if isinstance(self.parent, weakref.ref):
            return self.parent()
        return self.parent
    def __repr__(self):
        return f"Node({self.name})"

def refcount(obj):
    return sys.getrefcount(obj) - 1  # вычитаем временную ссылку в аргументе



# Цикл со сильными ссылками
a = Node("A")
b = Node("B", parent=a)
a.parent = b  # сильная ссылка: a -> b -> a (цикл)

print("Strong cycle refcounts:", refcount(a), refcount(b))

del a, b
gc.collect()   # объекты должны быть собраны как циклический мусор


# Цикл с weakref
c = Node("C")
d = Node("D", parent=c, weak_parent=True)  # d -> c (слабая)
c.parent = d  # c -> d (сильная)

print("Weak cycle refcounts:", refcount(c), refcount(d))

wr_c = weakref.ref(c)
print("wr_c before:", wr_c())

del c
gc.collect()

print("wr_c after deletion:", wr_c())

# Сильный цикл:
# - Оба объекта имеют refcount >= 1 из-за цикла
# - После del a, b: refcount не падает до 0
# - GC собирает их как циклический мусор

# Слабый цикл:
# - Слабая ссылка не увеличивает refcount
# - После удаления c: refcount(c)=0 → объект удаляется
# - d.parent становится None
# - Нет циклического мусора, GC не нужен

Strong cycle refcounts: 3 3
Weak cycle refcounts: 2 3
wr_c before: Node(C)
wr_c after deletion: None
